In [1]:
import mediapy as media
import mujoco
import numpy as np


In [16]:
FPS=30
HEIGHT=480
WIDTH=640

def rollout_video(model, data, n_steps=200, n_frames=100, fps=FPS, ctrl_fn=None,
                   width=WIDTH, height=HEIGHT):
    """Step sim, capture frames, return as video array.
    ctrl_fn(model, data, step_idx) -> None, called before each mj_step if provided.
    """
    renderer = mujoco.Renderer(model, height=height, width=width)
    frame_every = max(1, n_steps // n_frames)
    frames = []
    for i in range(n_steps):
        if ctrl_fn is not None:
            ctrl_fn(model, data, i)
        mujoco.mj_step(model, data)
        if i % frame_every == 0 and len(frames) < n_frames:
            renderer.update_scene(data)
            frames.append(renderer.render())
    renderer.close()
    return frames

In [23]:
model = mujoco.MjModel.from_xml_path("./data/scene.xml")
data = mujoco.MjData(model)

In [24]:
frames = rollout_video(model, data)
media.show_video(frames, fps=FPS)

This browser does not support the video tag.


In [15]:
print("nq:", model.nq, "nu:", model.nu)
assert not np.isnan(data.qpos).any(), "sim diverged"
print("ncon at rest:", data.ncon)
for i in range(data.ncon):
    c = data.contact[i]
    print(model.geom(c.geom1).name, model.geom(c.geom2).name)

nq: 23 nu: 16
ncon at rest: 3
floor 
floor 
floor 


# cpu ppo

In [27]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env